## Today {.smaller}

Last week built the 2017 transformer. **Today is one question: what had to
change to get from that to a model you can hold a conversation with?**

- **It got bigger**, in a specific ratio that the field got wrong at first.
- **Four parts were replaced**: how position is encoded, how it is normalized,
  how attention is stored, and what the MLP does.
- **It was trained differently**: one objective, a very large pile of text, and
  a set of tricks that exist only to keep the run from diverging.
- **It was aligned afterwards**, which is a separate training run and is most of
  what makes a chat model feel like one.

Then inference, which is where the money goes, and emergence, which is the part
you have all heard about and where most of what gets said is loose.

## Labs and quizzes {.smaller}

**Lab 8** was due yesterday: a transformer encoder block from `nn.Linear` and
`softmax`, checked against `nn.TransformerEncoderLayer`, then broken three ways
on purpose.

**Quiz 9** is at the end of class. It covers the Oct 22 lecture and Lab 8:
attention, query, key and value, the scaled dot product, multi-head, positional
encoding, the two masks, and the block. **Nothing from today is on it.**

**Lab 9** goes out today, due Wednesday Nov 4.

## What an LLM is {.smaller}

**A large language model is a decoder-only transformer trained on a very large
corpus to predict the next token, and nothing in its architecture is new.**

Every part was on last week's slides. What changed is scale, four component
swaps, and two extra training stages after the pretraining.

That is worth being blunt about, because the marketing is not. There is no new
mathematical idea between *Attention Is All You Need* and the model you used
this morning. There is a great deal of engineering.

# It got bigger

## Five and a half years of scaling {.smaller}

[![](images/fig-growth.png){.dw80}](images/fig-growth.png){target="_blank" .zoom}

From **65 million** parameters to **1.7 trillion**, about
**26,154 times** in under six years, or roughly
**5 times a year**. A straight line on a log axis is
exponential growth.

Published figures, not measured here, and GPT-4's is a reported estimate rather
than a disclosure. Notice that the line stops being the whole story around
2022, which is the next slide.

## Scaling laws {.smaller}

**A scaling law is an empirical relationship between how much you spend and how
well the model does.** Kaplan et al., 2020, found that test loss falls as a
power law in parameters, data and compute, with no sign of a floor.

That paper's practical reading was that parameters mattered most, and the field
acted on it: GPT-3 was 175 billion parameters trained on 300 billion tokens.

Hoffmann et al., 2022, redid the experiment over a wider range and found
something different. **For a fixed compute budget, parameters and training
tokens should grow together**, at roughly

$$D \approx 20\,N, \qquad \text{training FLOPs} \approx 6\,N\,D$$

for $N$ parameters and $D$ training tokens.

## Most of the famous models were undertrained {.smaller}

[![](images/fig-chinchilla.png){.dw95}](images/fig-chinchilla.png){target="_blank" .zoom}

GPT-3 saw **1.7** tokens per parameter where the compute-optimal
ratio is about **20**. The same compute spent on the line would have
trained a **51 billion** parameter model on far more data, and
that model would have been better *and* cheaper to serve.

Chinchilla itself was the demonstration: 70 billion parameters, 1.4 trillion
tokens, and it beat the 280 billion parameter Gopher trained on the same budget.

## Why nobody trains compute-optimal any more {.smaller}

Chinchilla optimises **training** cost. Nobody trains a model once and stops;
they serve it to millions of people, and serving cost scales with parameters,
not with how many tokens it was trained on.

So the modern move is to go deliberately past the optimum: take a smaller model
and train it far longer than Chinchilla would say. LLaMA-3-70B saw about
**214** tokens per parameter, ten times the compute-optimal ratio.

**The ratio you want depends on what you are paying for.** Compute-optimal if
you train once and throw it away; heavily overtrained if you have to serve it.

# Four parts got replaced

## Absolute position does not survive a long context {.smaller}

Week 9 added a positional vector to each embedding, either sinusoidal or
learned. Both break as context grows.

- **A learned table has a fixed number of rows.** Position 2,049 of a 2,048 row
  table does not exist, so you cannot run on a longer sequence at all.
- **Sinusoidal encoding is defined everywhere**, but the model has only ever
  seen the pattern for positions it trained on, and it does not reliably
  generalize past them.
- Both encode **where a token is**, when attention almost always wants to know
  **how far apart two tokens are**.

The fix is to stop adding position to the embedding and start applying it to
the query and the key instead.

## Rotary position embedding {.smaller}

**RoPE rotates each pair of dimensions in the query and key by an angle
proportional to the position.** Rotating two vectors by the same amount does
not change the angle between them, so a dot product of two rotated vectors
depends on their positions only through the difference.

[![](images/fig-rope.png){.dw95}](images/fig-rope.png){target="_blank" .zoom}

The right panel is the measurement. Holding the gap fixed and sliding the pair
along the sequence, the rotary score changes by **5e-6**, which is
floating point noise. The absolute encoding changes by **21.4**
over the same slide. RoPE is in LLaMA, Mistral, Qwen and most of what is current.

## RoPE in code {.smaller .codetight}

In [1]:
import torch

torch.manual_seed(6600)
d, base = 64, 10_000.0


def rope(x, pos):
    """Rotate each pair of dimensions by an angle proportional to position."""
    half = d // 2
    inv = base ** (-torch.arange(half, dtype=torch.float32) * 2 / d)
    ang = pos[:, None] * inv[None, :]
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * ang.cos() - x2 * ang.sin(),
                      x1 * ang.sin() + x2 * ang.cos()], dim=-1)


q, k = torch.randn(d), torch.randn(d)
print("the score depends only on the gap, not the position:")
for a, b in [(0, 3), (10, 13), (40, 43), (100, 103)]:
    qa = rope(q[None, :], torch.tensor([float(a)]))
    kb = rope(k[None, :], torch.tensor([float(b)]))
    print(f"  positions {a:>3} and {b:>3}  (gap 3)   score {float(qa @ kb.T):+.6f}")

print()
print("a different gap gives a different score, as it should:")
for a, b in [(0, 7), (40, 47)]:
    qa = rope(q[None, :], torch.tensor([float(a)]))
    kb = rope(k[None, :], torch.tensor([float(b)]))
    print(f"  positions {a:>3} and {b:>3}  (gap 7)   score {float(qa @ kb.T):+.6f}")

the score depends only on the gap, not the position:
  positions   0 and   3  (gap 3)   score +1.367940
  positions  10 and  13  (gap 3)   score +1.367940
  positions  40 and  43  (gap 3)   score +1.367940
  positions 100 and 103  (gap 3)   score +1.367938

a different gap gives a different score, as it should:
  positions   0 and   7  (gap 7)   score -0.705606
  positions  40 and  47  (gap 7)   score -0.705608


## Normalization: LayerNorm to RMSNorm {.smaller}

Week 9 showed why sequence models normalize over features. RMSNorm keeps that
and drops the mean:

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma}\,g + b, \qquad
  \text{RMSNorm}(x) = \frac{x}{\sqrt{\tfrac{1}{d}\sum_i x_i^2}}\,g$$

No mean subtraction and no bias, so it is one pass over the features rather
than two. It performs about the same and is measurably cheaper, which at this
scale is reason enough.

The other change is **where** it goes. The original paper normalized after the
residual add; every large model now normalizes before the sublayer
(`norm_first=True`), because post-norm networks of 50 or more blocks are
unstable to train.

## Attention: the KV cache, and then shrinking it {.smaller}

Generation is one token per forward pass, and every new token attends to every
previous one. Recomputing the keys and values of the whole prefix each step
makes generating $T$ tokens cost $O(T^2)$.

**The KV cache stores the projected keys and values and appends one column per
step**, so each step is $O(1)$ of projection plus one attention against the
store. It is not an optimization you can skip; it is how generation is done.

The cache then becomes the problem. It is per sequence, per layer, and grows
with the conversation.

## What the cache costs {.smaller}

[![](images/fig-kvcache.png){.dw95}](images/fig-kvcache.png){target="_blank" .zoom}

A 7 billion parameter model at a 32,000 token context needs about
**17 GB** of cache for **one** sequence with ordinary multi-head
attention. Serving many users at once, that, not the weights, is what fills the
GPU.

**Grouped query attention** is the fix: keep all the query heads, share keys and
values across groups of them. Eight key-value heads instead of 32 brings that
32k cache to **4.3 GB**. The right panel measures why the cache exists
at all: at **1,024** tokens of context, producing one more token costs
**10 times** less with the cache than without.

## The MLP: GELU to SwiGLU {.smaller}

Week 9's feed forward network was two linear layers with a nonlinearity,
widening to $4d$ and back. Modern models use a **gated** variant:

$$\text{SwiGLU}(x) = \big(\text{Swish}(xW_1) \odot xW_3\big)\,W_2$$

Three matrices instead of two. One branch computes values and the other
computes a gate that scales them elementwise, which is the same idea as an
LSTM's gates applied per position instead of per timestep.

Because there are three matrices, the hidden width is usually cut to about
$\tfrac{8}{3}d$ to keep the parameter count the same. **The FFN is about two
thirds of a transformer's parameters**, so this is not a small part of the model.

## Where the four changes land {.smaller}

| part | 2017 | now | why |
|---|---|---|---|
| position | added to the embedding | **RoPE**, rotating Q and K | relative, and extends past the trained length |
| normalization | LayerNorm, after the residual | **RMSNorm**, before the sublayer | cheaper, and stable at depth |
| attention | recompute each step | **KV cache**, then **grouped query** | generation would be quadratic; the cache then has to shrink |
| the MLP | two matrices and GELU | **SwiGLU**, three matrices | better at equal parameter count |

None of these change what a transformer is. Attention, multi-head, the
residuals, the causal mask and the block structure are all exactly last week's.

# How it is trained

## One objective {.smaller}

**Causal language modelling: predict the next token, for every position at
once.**

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T}
  \log p_\theta\!\left(x_t \mid x_{<t}\right)$$

Which is cross entropy with the next token as the label, and the causal mask
from last week is what makes computing all $T$ terms in one pass legitimate.

That is the whole pretraining objective. No labels, no human input, nothing
task-specific. The text is its own supervision, which is why it scales: the
limit is how much text you can find, not how much you can afford to annotate.

## Training data {.smaller}

Modern pretraining corpora are measured in trillions of tokens, assembled from
web crawl, code, books, and reference text. The work is almost entirely in what
you remove:

- **deduplication**, exact and near. Duplicated text gets memorized rather than
  learned, and it inflates your benchmark scores.
- **quality filtering**, usually a classifier trained to recognise text that
  looks like a curated source.
- **decontamination**, removing anything that appears in the evaluation sets.
  This one is routinely done badly, which is why benchmark numbers should be
  read with suspicion.

**Data work is most of the job and almost none of the papers.** The architecture
has barely moved; the corpora have moved enormously.

## Tokenization at this scale {.smaller}

Week 8's subword tokenizer, with the vocabulary size now a real engineering
decision. Typical modern vocabularies run 32,000 to 256,000.

A larger vocabulary means **shorter sequences**, which is quadratically cheaper
in attention, and **a bigger embedding table and output layer**, which is
linearly more parameters and more softmax work. Somewhere in the middle is the
right answer, and it depends on the languages you care about.

The costs show up in odd places. A tokenizer fitted mostly to English spends
several tokens per word on other languages, so the same sentence costs more to
process and more to buy. Numbers and code are similarly sensitive to how digits
and whitespace were split.

## Keeping a long run alive {.smaller}

A training run of this size is one job for weeks on thousands of accelerators,
and the practical content of a technical report is mostly about not losing it.

- **Learning rate**: a linear warmup over the first fraction of a percent of
  steps, then a cosine decay. Warmup exists because the first updates at full
  learning rate reliably diverge.
- **Gradient clipping** by global norm, usually at 1.0.
- **bf16** rather than fp16: the same 16 bits, more exponent and less mantissa,
  which matters because the failure mode at this scale is overflow.
- **Checkpoint often**, because hardware fails and you will be restarting from
  a checkpoint, not from zero.

## How the work is split {.smaller}

One model no longer fits on one device, so the work is cut up. You should
recognize the names; you will not be asked to implement any of them.

| kind | what is split | cost |
|---|---|---|
| **data parallel** | the batch, model replicated | memory, every replica holds everything |
| **tensor parallel** | individual matrices, across devices | communication inside every layer |
| **pipeline parallel** | whole layers, across devices | devices idle waiting for each other |
| **ZeRO / FSDP** | the optimizer state, gradients and parameters | communication, in exchange for a much smaller footprint |

The thing worth taking away: **optimizer state is the memory problem, not
weights.** Adam keeps two moments per parameter, so training a model costs
several times what storing it does.

# Making it useful

## A pretrained model is not an assistant {.smaller}

A model trained only to predict the next token does exactly that. Ask one a
question and a plausible continuation is **another question**, because that is
what documents full of questions look like.

Nothing is wrong with the model. It was never trained to be helpful, and
helpfulness is not a property of text statistics.

Two further stages close that gap, and together they are what separates a base
model from a chat model.

[![](images/fig-stages.png){.dw95}](images/fig-stages.png){target="_blank" .zoom}

## Supervised fine-tuning {.smaller}

**SFT is ordinary supervised training on demonstrations: prompt and response
pairs written or curated by people, with the loss taken on the response only.**

The objective is unchanged from pretraining; only the data changed. That is why
it is cheap, typically thousands to hundreds of thousands of examples rather
than trillions of tokens.

SFT is what installs the shape of a conversation: that a question gets an
answer, that instructions get followed, that there is a turn structure at all.
It does not install judgement about which of two decent answers is better,
because a demonstration gives one answer and no comparison.

## Learning from preferences {.smaller}

The next stage uses **comparisons** instead of demonstrations: given one prompt
and two responses, a human says which is better. Comparing is far easier than
writing, so this data is cheaper to collect and more consistent.

**RLHF** turns those comparisons into a *reward model* that scores responses,
then optimizes the language model against that score with reinforcement
learning, usually PPO, with a penalty for drifting too far from the SFT model.

**DPO** skips the reward model. The same preference data is turned directly
into a loss on the language model, which is simpler, has fewer moving parts and
performs comparably. Much of the field moved to it or a variant.

You should be able to say what each is for. You will not be asked to derive
either objective.

## What alignment does and does not do {.smaller}

**Does.** Make it follow instructions, adopt a consistent format and register,
refuse some categories of request, and prefer answers people rate highly.

**Does not.** Make it know anything it did not learn in pretraining, or make it
accurate. Those are pretraining properties, and no amount of preference data
adds a fact that was never in the corpus.

**And the failure mode is specific.** Optimizing against a learned reward model
rewards answers that *score* well, which is not the same as answers that *are*
good. Confident, fluent, agreeable and wrong scores well. That is most of what
people mean by an LLM being sycophantic, and it is a predictable consequence of
the training objective rather than a mystery.

# Inference

## How the next token is chosen {.smaller}

[![](images/fig-decoding.png){.dw95}](images/fig-decoding.png){target="_blank" .zoom}

A real distribution out of GPT-2. The top token is **Paris**, and it is
correct, but it has only **0.07** of the probability mass.

**Temperature** divides the logits before the softmax: below 1 sharpens toward
the top token, above 1 flattens. **Top-p** keeps the smallest set of tokens
whose probability sums to $p$ and renormalizes, so the number kept adapts to how
confident the model is: 52 tokens at $p = 0.5$ here, **1,018** at
$p = 0.9$, out of 50,257.

Greedy decoding, always taking the top token, is temperature 0. It is right for
extraction and wrong for anything that should read naturally.

## Sampling in code {.smaller .codetight}

In [2]:
import torch

torch.manual_seed(6600)
# a stand-in for one real next-token distribution: a few likely tokens and a
# long tail, which is the shape the GPT-2 figure showed
logits = torch.cat([torch.tensor([6.0, 5.4, 5.1, 4.4]), torch.randn(996) - 1])


def sample(logits, temperature=1.0, top_p=1.0):
    p = (logits / temperature).softmax(-1)
    order = p.argsort(descending=True)
    keep = (p[order].cumsum(0) - p[order]) < top_p      # always keeps the first
    p = torch.zeros_like(p).scatter_(0, order[keep], p[order][keep])
    return p / p.sum()


for temp in (0.2, 1.0, 1.6):
    p = sample(logits, temperature=temp)
    print(f"  temperature {temp:<4} top token {p.max():.3f}   "
          f"tokens above 1%: {(p > 0.01).sum():>3}")
print()
for tp in (0.5, 0.9, 1.0):
    p = sample(logits, top_p=tp)
    print(f"  top-p {tp:<5} keeps {(p > 0).sum():>4} of {len(logits)} tokens")

  temperature 0.2  top token 0.942   tokens above 1%:   3
  temperature 1.0  top token 0.268   tokens above 1%:   5
  temperature 1.6  top token 0.055   tokens above 1%:   4

  top-p 0.5   keeps    3 of 1000 tokens
  top-p 0.9   keeps  370 of 1000 tokens
  top-p 1.0   keeps 1000 of 1000 tokens


## Why serving is expensive {.smaller}

Conflating these two is the most common mistake in capacity planning.

[![](images/fig-serve.png){.dw95}](images/fig-serve.png){target="_blank" .zoom}

**Decode speed tracks memory bandwidth, not FLOPs.** A faster GPU with the same
memory bandwidth generates at nearly the same rate.

## Making it cheaper {.smaller}

| technique | what it does | what it costs |
|---|---|---|
| **quantization** | store weights in 8 or 4 bits instead of 16 | a little quality, usually surprisingly little |
| **grouped query attention** | shrink the KV cache | a little quality |
| **continuous batching** | start new requests as others finish | nothing, it is scheduling |
| **paged attention** | page the cache like virtual memory | nothing, it is bookkeeping |
| **speculative decoding** | a small model drafts, the big one checks in parallel | nothing, output is identical |

Speculative decoding is the elegant one. A draft model proposes several tokens,
the real model verifies them in a single forward pass, and any prefix it agrees
with is kept. The output distribution is provably unchanged.

## Prompting, in one slide {.smaller}

[![](images/chain-of-thought.png){.dw60}](images/chain-of-thought.png){target="_blank" .zoom}

Everything the model will use is in its context window, so prompting is just
deciding what goes there. **Chain of thought** is asking for the reasoning
before the answer, which helps because each generated token is another forward
pass: writing the steps out gives the model more computation to spend.

Retrieval does the same thing with facts rather than reasoning: find the
relevant text, put it in the context, and the model reads it rather than
recalling it. **Nothing here changes the weights.**

## Emergent capabilities {.smaller}

**An emergent capability is one that is near chance at smaller scales and then
rises sharply past some size**, rather than improving smoothly. Arithmetic,
multi-step reasoning and following an unfamiliar instruction format are the
usual examples.

The honest caveat matters as much as the claim. Several of these curves look
like step functions **because the metric is a step function**: exact-match
scoring gives no credit for being nearly right, so smooth underlying
improvement shows up as a sudden jump. Change to a graded metric and some
emergence disappears.

Some of it survives that correction and some does not. Treat a claimed
emergent ability as a claim about the metric until you have checked.

## Evaluating these is genuinely hard {.smaller}

Benchmarks are cheap to game and easy to contaminate, and the two interact: if
a test set was on the web before the crawl, a high score may be memorization.

What tends to survive contact with reality: **held-out perplexity** on text you
collected after the model shipped, **task suites you built yourself** from your
own data, and **human comparison**, which is expensive and still the thing
everyone falls back on.

The practical version for your project: **do not trust a published score on a
public benchmark as evidence that a model will work on your problem.** Build a
small evaluation set of your own, by hand if necessary, and use that.

# Wrap-up

## Wrap-up {.smaller}

- An LLM is a **decoder-only transformer**. Nothing in the architecture is new
  since 2017.
- **Scaling laws** say parameters and tokens should grow together, about 20
  tokens per parameter, and most famous models were well off that line.
- Modern serving cost pushes **past** compute-optimal on purpose.
- Four parts were replaced: **RoPE**, **RMSNorm before the sublayer**, **the KV
  cache plus grouped query attention**, and **SwiGLU**.
- Pretraining is one objective, next-token prediction, on a corpus whose
  construction is most of the work.
- **SFT** installs the format; **preference optimization** installs the
  judgement. Neither adds knowledge.
- Generation is **memory-bound**, which is why the KV cache governs what you
  can serve.
- Treat **emergence** and **benchmark scores** as claims about a metric until
  checked.

## Supplemental content {.smaller}

Optional. Prof. Hickman's [centralized lecture
content](https://jfh.georgetown.domains/centralized-lecture-content/):

- [Large language models](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/llm/notes.html),
  which goes considerably deeper than today on every section
- [Running an LLM locally](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/running-and-LLM-locally/notes.html)
- [Agentic AI](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/agentic-ai/notes.html)

Also optional, and all three are short: Kaplan et al.,
[Scaling Laws](https://arxiv.org/abs/2001.08361) (2020); Hoffmann et al.,
[Training Compute-Optimal LLMs](https://arxiv.org/abs/2203.15556) (2022), the
Chinchilla paper; and Schaeffer et al.,
[Are Emergent Abilities a Mirage?](https://arxiv.org/abs/2304.15004) (2023),
which is the metric argument from two slides ago.

## Lab 9 and Quiz 9 {.smaller}

**Lab 9** is out today, due Wednesday Nov 4. You take a pretrained GPT-2, look
at what it actually predicts, implement temperature and top-p sampling
yourself, measure what the KV cache saves, and compare a base model against an
instruction-tuned one on the same prompts.

**Quiz 9** is now. Scope is the Oct 22 lecture and Lab 8 only: attention, Q, K
and V, multi-head, positional encoding, the masks, and the block. A formula
sheet comes with it.

Next week is **Generative Modeling**: autoencoders, latent spaces and VAEs.